# 应用结构

一个 LangGraph 应用由以下部分组成: 一个或多个 graph, 一个配置文件 (`langgraph.json`), 一个用于指定依赖的文件, 以及一个可选的、用于指定环境变量的 `.env` 文件。

本指南展示应用的典型结构, 并演示如何提供所需的配置, 以便用 [LangSmith Deployment](https://docs.langchain.com/langsmith/deployment) 部署应用。

> **提示**
>
> LangSmith Deployment 是一个托管平台, 用于部署并扩展 LangGraph agent。它负责处理基础设施、扩展与运维相关的事项, 让你可以直接从代码仓库部署有状态、长时运行的 agent。更多内容见 [Deployment 文档](https://docs.langchain.com/langsmith/deployment)。

## 核心概念

若要使用 LangSmith 部署, 应提供以下信息:

1. 一个 [LangGraph 配置文件](#configuration-file-concepts) (`langgraph.json`), 用于指定应用要使用的依赖、graphs 与环境变量。
2. 实现应用逻辑的 [graphs](#graphs)。
3. 一个文件, 用于指定运行应用所需的[依赖](#dependencies)。
4. 运行应用所需的[环境变量](#environment-variables)。

## 文件结构

下面是应用目录结构的示例:

## Python (requirements.txt)

```plaintext
my-app/
├── my_agent # all project code lies within here
│   ├── utils # utilities for your graph
│   │   ├── __init__.py
│   │   ├── tools.py # tools for your graph
│   │   ├── nodes.py # node functions for your graph
│   │   └── state.py # state definition of your graph
│   ├── __init__.py
│   └── agent.py # code for constructing your graph
├── .env # environment variables
├── requirements.txt # package dependencies
└── langgraph.json # configuration file for LangGraph
```

## Python (pyproject.toml)

```plaintext
my-app/
├── my_agent # all project code lies within here
│   ├── utils # utilities for your graph
│   │   ├── __init__.py
│   │   ├── tools.py # tools for your graph
│   │   ├── nodes.py # node functions for your graph
│   │   └── state.py # state definition of your graph
│   ├── __init__.py
│   └── agent.py # code for constructing your graph
├── .env # environment variables
├── langgraph.json  # configuration file for LangGraph
└── pyproject.toml # dependencies for your project
```

> **注意**
>
> LangGraph 应用的目录结构会因所用的编程语言与包管理器而有所不同。

## 配置文件

`langgraph.json` 文件是一个 JSON 文件, 用于指定部署 LangGraph 应用所需的依赖、graphs、环境变量以及其他设置。

关于 JSON 文件中所有受支持的键的细节, 见 [LangGraph 配置文件参考](https://docs.langchain.com/langsmith/cli#configuration-file)。

> **提示**
>
> [LangGraph CLI](https://docs.langchain.com/langsmith/cli) 默认使用当前目录下的配置文件 `langgraph.json`。

### 示例

* 依赖包括一个自定义的本地包与 `langchain_openai` 包。
* 会从文件 `./your_package/your_file.py` 中加载一个 graph, 使用的是变量 `variable`。
* 环境变量从 `.env` 文件加载。

```json
{
  "dependencies": ["langchain_openai", "./your_package"],
  "graphs": {
    "my_agent": "./your_package/your_file.py:agent"
  },
  "env": "./.env"
}
```

## 依赖

LangGraph 应用可能依赖其他 Python 包。

为了让依赖被正确设置, 你通常需要指定以下信息:

1. 该目录中一个用于指定依赖的文件 (例如 `requirements.txt`、`pyproject.toml` 或 `package.json`)。
2. [LangGraph 配置文件](#configuration-file-concepts)中的一个 `dependencies` 键, 用于指定运行 LangGraph 应用所需的依赖。
3. 任何额外的二进制文件或系统库都可以通过 [LangGraph 配置文件](#configuration-file-concepts)中的 `dockerfile_lines` 键来指定。

## Graphs

使用 [LangGraph 配置文件](#configuration-file-concepts)中的 `graphs` 键来指定哪些 graphs 会在部署后的 LangGraph 应用中可用。

你可以在配置文件中指定一个或多个 graphs。每个 graph 都由一个名称 (应当是唯一的) 与一个路径来标识, 该路径指向以下两者之一: (1) 编译后的 graph, 或 (2) 一个用于构建 graph 的函数。

## 环境变量

如果你在本地使用已部署的 LangGraph 应用, 可以在 [LangGraph 配置文件](#configuration-file-concepts)的 `env` 键中配置环境变量。

对于生产部署, 你通常希望在部署环境中配置环境变量。